# IPUMS [NHGIS](https://www.nhgis.org) Data Extraction Using [ipumsr](https://cran.r-project.org/web/packages/ipumsr/index.html) - Supplemental Exercise 1
### by [Kate Vavra-Musser](https://vavramusser.github.io) for the [R Spatial Notebook Series](https://vavramusser.github.io/r-spatial)

## Introduction
This notebook provides an additional example of the IPUMS NHGIS data extraction process using the IPUMS API via the ipumsr R package.  This exercise is a supplement to the workflow introducted in Chapter 3.4 IPUMS NHGIS Data Extraction Using ipumsr.

### Notebook Goals
This notebook replicates the IPUMS NHGIS data extraction process and extracts a NHGIS point dataset on population of populated places and point locations shapefile.  The resulting data file is used in subsequent notebooks in the R Spatial Notebooks series.  The notebook provides an example of extracting point-based spatial data with attached attribute data from the IPUMS NHGIS repository.

### ✨ Prerequisites ✨
* Complete [Introduction to IPUMS and the IPUMS API](https://platform.i-guide.io/notebooks/82d3b176-e4e6-4307-8186-318a3fe6c81a)
* Set Up Your [IPUMS Account and API Key](https://account.ipums.org/api_keys)
* Complete [Introduction to sf: Reading, Writing, and Inspecting Vector Data](https://platform.i-guide.io/notebooks/9968babe-22e4-4c3d-98e2-d8b45e9672cd)
* Complete [IPUMS NHGIS Data Extraction Using ipumsr](https://platform.i-guide.io/notebooks/be08e56e-1c08-458e-a230-263c64d386bc)

### Notebook Overview
1. Setup
2. Extraction Workflow: Point-Based Shapefiles + Tabular Data

---

## 1. Setup
This section will guide you through the process of installing essential packages and setting your IPUMS API key.

#### Required Packages

[**dplyr**](https://cran.r-project.org/web/packages/dplyr/index.html) A Grammar of Data Manipulation. This notebook uses the the following functions from *dplyr*.

* [*filter*](https://rdrr.io/cran/dplyr/man/filter.html) · keep rows that match a condition

[**ipumsr**](https://cran.r-project.org/web/packages/ipumsr/index.html) An R Interface for Downloading, Reading, and Handling IPUMS Data.  This notebook uses the the following functions from *ipumsr*.

* [*define_extract_nhgis*](https://rdrr.io/cran/ipumsr/man/define_extract_nhgis.html) · define an IPUMS NHGIS extract request
* [*download_extract*](https://rdrr.io/cran/ipumsr/man/download_extract.html) · download a completed IPUMS data extract
* [*get_metadata_nhgis*](https://rdrr.io/cran/ipumsr/man/get_metadata_nhgis.html) · list available data sources from IPUMS NHGIS
* [*read_ipums_sf*](https://rdrr.io/cran/ipumsr/man/read_ipums_sf.html) · read spatial data from an IPUMS extract
* [*read_nhgis*](https://rdrr.io/cran/ipumsr/man/read_nhgis.html) · read tabular data from an NHGIS extract
* [*set_ipums_api_key*](https://rdrr.io/cran/ipumsr/man/set_ipums_api_key.html) · set your IPUMS API key
* [*submit_extract*](https://rdrr.io/cran/ipumsr/man/submit_extract.html) · submit an extract request via the IPUMS API
* *tst_spec* · create a *tst_spec* object containing a time series table specification
* [*wait_for_extract*](https://rdrr.io/cran/ipumsr/man/wait_for_extract.html) · wait for an extract to finish processing

[**purrr**](https://cran.r-project.org/web/packages/purrr/index.html) A complete and consistent functional programming toolkit for R. This notebook uses the the following functions from *purrr*.

* [*map()*](https://rdrr.io/cran/purrr/man/map.html) and [*map_dfr()*](https://rdrr.io/cran/purrr/man/map_dfr.html) · apply a function to each element of a vector

[**sf**](https://cran.r-project.org/web/packages/sf/index.html) Support for simple features, a standardized way to encode spatial vector data. Binds to 'GDAL' for reading and writing data, to 'GEOS' for geometrical operations, and to 'PROJ' for projection conversions and datum transformations. Uses by default the 's2' package for spherical geometry operations on ellipsoidal (long/lat) coordinates.  This notebook uses the following functions from *sf*.

* [*st_write*](https://rdrr.io/cran/sf/man/st_write.html) · Write simple features object to file or database

### 1a. Install and Load Required Packages
If you have not already installed the required packages, uncomment and run the code below:

In [ ]:
# install.packages(c("dplyr", "ipumsr", "purr", "sf"))

Load the packages into your workspace.

In [2]:
library(dplyr)
library(ipumsr)
library(purrr)
library(sf)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Linking to GEOS 3.10.2, GDAL 3.4.1, PROJ 7.2.1; sf_use_s2() is TRUE



### 1b. Set Your IPUMS API Key

Store your [IPUMS API key](https://account.ipums.org/api_keys) in your environment using the following code.

Refer to [Chapter 1.1 Introduction to IPUMS and the IPUMS API](https://platform.i-guide.io/notebooks/82d3b176-e4e6-4307-8186-318a3fe6c81a) for instructions on setting up your IPUMS account and API key.

In [3]:
ipumps_api_key = readline("Please enter your IPUMS API key: ")
set_ipums_api_key(ipumps_api_key, save = T, overwrite = T)

Existing .Renviron file copied to C:\Users\vavra\Documents/.Renviron_backup for backup purposes.

The environment variable IPUMS_API_KEY has been set and saved for future sessions.



## 2. NHGIS Time-Series + Points

### 2a. View and Filter the List of Time-Series Datasets

First we will take a look at the list of available time-series datasets in the NHGIS repository that fit our critera.  We are looking for datasets on *total population* at the *place* geography.

In [4]:
metadata_datts_filter <- get_metadata_nhgis("time_series_tables") %>%
    filter(grepl("place", geog_levels, ignore.case = T), grepl("total population", description, ignore.case = T)) %>%
    select(name, description) %>%
    as.data.frame() %>%
    print()

Warning message:
"`get_metadata_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `get_metadata_catalog()` to obtain summary metadata or
  `get_metadata()` to obtain detailed metadata."


  name      description
1  AV0 Total Population
2  CL8 Total Population


The results returned three possible datasets, *AV0*, *B78*, and *CL8*.

### 2b. Identify Available Years and Geographic Levels

Next let's take a look at the available years and geographic levels for each of these three datasets.

In [5]:
# get metadata for each time-series table
metadata_list <- map(metadata_datts_filter$name, ~ get_metadata_nhgis(time_series_table = .x))

# combine into a data frame with the necessary columns
metadata_combined <- map_dfr(metadata_list, function(metadata) {
  data.frame(
    name = metadata$name,
    description = metadata$description,
    # extract only the "description" column from the nested tibbles in "years" and "geog_levels"
    years = paste(metadata$years$description, collapse = ", "),
    geog_levels = paste(metadata$geog_levels$name, collapse = ", ")
  )
})

# print the final data frame
metadata_combined

name,description,years,geog_levels
<chr>,<chr>,<chr>,<chr>
AV0,Total Population,"1970, 1980, 1990, 2000, 2010, 2006-2010, 2007-2011, 2008-2012, 2009-2013, 2010-2014, 2011-2015, 2012-2016, 2013-2017, 2014-2018, 2015-2019, 2020, 2016-2020, 2017-2021, 2018-2022, 2019-2023","nation, region, division, state, county, tract, cty_sub, place"
CL8,Total Population,"1990, 2000, 2010, 2020","state, county, tract, blck_grp, cty_sub, place, cd111th, cbsa, urb_area, zcta"


Let's choose the *CL8* time-series table and move onto the next step, selecting a shapefile to go with our time-series data.

### 2c. View and Filter the List of Geography Shapefiles

Since we want our data to be at the *place (points)* geography, let's filter the shapfile metadata to only include shapfiles which include the word *points* in the description of their *geographic_level*.

In [6]:
metadata_shp <- get_metadata_nhgis("shapefiles") %>%
    filter(grepl("points", geographic_level, ignore.case = T)) %>%
    print(n = Inf)

# A tibble: 18 x 6
   name                       year  geographic_level extent       basis sequence
   <chr>                      <chr> <chr>            <chr>        <chr>    <int>
 1 us_place_point_1900_tlgnis 1900  Place (Points)   United Stat~ GNIS~       37
 2 us_place_point_1910_tlgnis 1910  Place (Points)   United Stat~ GNIS~       43
 3 us_place_point_1920_tlgnis 1920  Place (Points)   United Stat~ GNIS~       49
 4 us_place_point_1930_tlgnis 1930  Place (Points)   United Stat~ GNIS~       55
 5 us_place_point_1940_tlgnis 1940  Place (Points)   United Stat~ GNIS~       61
 6 us_place_point_1950_tlgnis 1950  Place (Points)   United Stat~ GNIS~       67
 7 us_place_point_1960_tlgnis 1960  Place (Points)   United Stat~ GNIS~       74
 8 us_place_point_1970_tlgnis 1970  Place (Points)   United Stat~ GNIS~       81
 9 us_place_point_1980_tlgnis 1980  Place (Points)   United Stat~ GNIS~       91
10 us_place_point_1990_tlgnis 1990  Place (Points)   United Stat~ GNIS~      102
11 us_pla

Let's select the 2010 shapefile (*us_place_point_2010_tlgnis*).

### 2d. Time-Series + Shapefile Extraction Specification and Submission

Now that we have selected our time-series dataset (*CL8*), the geographic level for our time-series dataset (*place*), and our shapefile (*us_place_point_2010_tlgnis*) we are ready to define and submit our exraction request to the IPUMS API.

In [7]:
extract_definition <- define_extract_nhgis(description = "I-GUIDE NHGIS Places Points Extraction",
                                           time_series = tst_spec(name = "CL8",
                                                                  geog_levels = "place"),
                                           shapefiles = "us_place_point_2010_tlgnis")

Warning message:
"`define_extract_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `define_extract_agg()` instead."


Submitting the extraction definition object *extract_definition* to the API.

In [8]:
extraction_submitted <- submit_extract(extract_definition)
extraction_complete <- wait_for_extract(extraction_submitted)
extraction_complete$status
filepath <- download_extract(extraction_submitted, overwrite = T)

Successfully submitted IPUMS NHGIS extract number 119

Checking extract status...

Waiting 10 seconds...

Checking extract status...

IPUMS NHGIS extract 119 is ready to download.



[1] "completed"

  |======================================================================| 100%
  |======================================================================| 100%


Data file saved to C:/Users/vavra/Dropbox/R Spatial/r-spatial/03 IPUMS Data Acquisition and Extraction/nhgis0119_csv.zip
Shapefile saved to C:/Users/vavra/Dropbox/R Spatial/r-spatial/03 IPUMS Data Acquisition and Extraction/nhgis0119_shape.zip



The result of the extraction request will be two files 1) a time-series table containing the populationd data and 2) the places (points) geography shapefile.  The next step is to read these two files into R.

In [9]:
dat <- read_nhgis(filepath[1])
shp <- read_ipums_sf(filepath[2])

Warning message:
"`read_nhgis()` was deprecated in ipumsr 0.9.0.
i Please use `read_ipums_agg()` instead."
Use of data from IPUMS NHGIS is subject to conditions including that users should cite the data appropriately. Use command `ipums_conditions()` for more details.

Rows: 29261 Columns: 16
-- Column specification --------------------------------------------------------
Delimiter: ","
chr  (5): GISJOIN, STATE, STATEA, PLACE, PLACEA
dbl (11): GEOGYEAR, CL8AA1990, CL8AA1990L, CL8AA1990U, CL8AA2000, CL8AA2000L...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


Let's take a look at the dimesions of the time-series data (*dat*) and the shapefile (*shp*).

In [10]:
dim(dat)
dim(shp)

[1] 29261    16

[1] 29514     8

The time-series table includes 16 variables for 29,261 places and the shapefile includes 8 attributes for 29,514 places.  The number of places represented by the time-series table is slightly smaller than the number of places represented in the shapefile.  It could be that some of the places represented in the shapefile do not have population counts available in the data table.

Let's take a look at the first few lines of the time-series table on population and the places (points) shapefile.

In [11]:
head(dat)

GISJOIN,GEOGYEAR,STATE,STATEA,PLACE,PLACEA,CL8AA1990,CL8AA1990L,CL8AA1990U,CL8AA2000,CL8AA2000L,CL8AA2000U,CL8AA2010,CL8AA2020,CL8AA2020L,CL8AA2020U
<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
G01000100,2010,Alabama,01,Abanda CDP,00100,177.25,120,213,131.68,89,153,192,133.00,133,133
G01000124,2010,Alabama,01,Abbeville city,00124,3176.41,2996,3186,2987.18,2839,2991,2688,2358.00,2358,2358
G01000460,2010,Alabama,01,Adamsville city,00460,5620.12,2558,10483,5107.32,4963,7050,4522,4353.93,2274,6083
G01000484,2010,Alabama,01,Addison town,00484,715.01,278,1147,743.68,632,994,758,659.00,659,659
G01000676,2010,Alabama,01,Akron town,00676,444.66,284,575,448.09,365,545,356,225.00,225,225
G01000820,2010,Alabama,01,Alabaster city,00820,15885.55,11096,19519,24415.70,20176,27874,30352,32519.23,25856,34631


In [12]:
head(shp)

ERROR while rich displaying an object: Error in loadNamespace(x): there is no package called 'geojsonio'

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_geojson.sf(obj)
8. repr_geojson(geo

STATE,NHGISST,PLACE,NAME,NHGISPLACE,YEAR,GISJOIN,geometry
<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<POINT [m]>
Alabama,010,Abanda CDP,Abanda,G010001000,2010,G01000100,POINT (969164.2 -438781.6)
Alabama,010,Abbeville city,Abbeville,G010001240,2010,G01000124,POINT (1014139 -605274.1)
Alabama,010,Adamsville city,Adamsville,G010004600,2010,G01000460,POINT (832255.1 -396765.6)
Alabama,010,Addison town,Addison,G010004840,2010,G01000484,POINT (805355.9 -331802.2)
Alabama,010,Akron town,Akron,G010006760,2010,G01000676,POINT (767096.4 -483924.3)
Alabama,010,Alabaster city,Alabaster,G010008200,2010,G01000820,POINT (848916.6 -435175.1)


Before we join the *dat* and *shp* files, let's remove a few of data columns which are less useful for our analyses.

We will only keep the following attributes:

**From the *dat* Tabular Data**
* GIS Join Key (*GISJOIN*)
* State (*STATE*)
* Place (*PLACE*)
* 1990 Total Population (*CL8AA1990*)
* 2000 Total Population (*CL8AA2000*)
* 2010 Total Population (*CL8AA2010*)
* 2020 Total Population (*CL8AA2020*)

**From the *shp* Geography Data**
* GIS Join Key (*GISJOIN*)
* Name (*NAME*)

In [13]:
dat_cols <- c("GISJOIN", "PLACE", "STATE", "CL8AA1990", "CL8AA2000", "CL8AA2010", "CL8AA2020")
dat <- dat[dat_cols]

shp_cols <- c("GISJOIN", "NAME")
shp <- shp[shp_cols]

We kept the *GISJOIN* join key column in both files and we will use this common key to join the two datasets using the *merge* function.

In [14]:
# merge the time-series population data with the county geographic data
dat_shp <- merge(dat, shp, by = "GISJOIN")

Finally we will save the data in shapefile format.

In [15]:
st_write(dat_shp, "ipums_nhgis_places.shp", driver = "ESRI Shapefile", delete_dsn = T)

Warning message in CPL_write_ogr(obj, dsn, layer, driver, as.character(dataset_options), :
"GDAL Error 1: ipums_nhgis_places.shp does not appear to be a file or directory."


Deleting source `ipums_nhgis_places.shp' failed
Writing layer `ipums_nhgis_places' to data source 
  `ipums_nhgis_places.shp' using driver `ESRI Shapefile'
Writing 29261 features with 8 fields and geometry type Point.


At the end of this notebook we have saved a copy of the time-series table with population data from the 1990, 2000, 2010, and 2020 Decennial Censuses for populated places in the United States to the file *ipums_nhgis_places.csv* and a copy of the complementary geographic data file for populated places to the shapefile *ipums_nhgis_places.shp*.